In [ ]:
#Install and load necessary packages
install.packages(c("foreach", "doParallel", "MASS"))
library(foreach)
library(doParallel)
#library(tictoc) # for timing the simulation

# Define the core simulation function

mvrnorm <-
    function(n = 1, mu, Sigma, tol=1e-6, empirical = FALSE, EISPACK = FALSE)
{
    p <- length(mu)
    if(!all(dim(Sigma) == c(p,p))) stop("incompatible arguments")
    if(EISPACK) stop("'EISPACK' is no longer supported by R", domain = NA)
    eS <- eigen(Sigma, symmetric = TRUE)
    ev <- eS$values
    if(!all(ev >= -tol*abs(ev[1L]))) stop("'Sigma' is not positive definite")
    X <- matrix(rnorm(p * n), n)
    if(empirical) {
        X <- scale(X, TRUE, FALSE) # remove means
        X <- X %*% svd(X, nu = 0)$v # rotate to PCs
        X <- scale(X, FALSE, TRUE) # rescale PCs to unit variance
    }
    X <- drop(mu) + eS$vectors %*% diag(sqrt(pmax(ev, 0)), p) %*% t(X)
    nm <- names(mu)
    if(is.null(nm) && !is.null(dn <- dimnames(Sigma))) nm <- dn[[1L]]
    dimnames(X) <- list(nm, NULL)
    if(n == 1) drop(X) else t(X)
}

gen_two_normals_sameCV <- function(n = 10000,
                                   mu1 = 5, mu2 = 8,
                                   CV = 0.4,
                                   rho = 0.6,
                                   seed = 123) {
  stopifnot(mu1 > 0, mu2 > 0, CV >= 0, abs(rho) < 1)
  set.seed(seed)

  # standard deviations from CV
  sd1 <- CV * mu1
  sd2 <- CV * mu2

  # covariance matrix
  Sigma <- matrix(c(sd1^2, rho * sd1 * sd2,
                    rho * sd1 * sd2, sd2^2), 2, 2)

  # simulate bivariate normal (exact Pearson corr = rho)
  mat <- mvrnorm(n = n, mu = c(mu1, mu2), Sigma = Sigma)

  X <- mat[,1]; Y <- mat[,2]

  list(X = X, Y = Y,
       params = list(mu1 = mu1, sd1 = sd1, mu2 = mu2, sd2 = sd2, CV = CV, rho = rho),
       summary = list(mean_X = mean(X), sd_X = sd(X), CV_X = sd(X)/mean(X),
                      mean_Y = mean(Y), sd_Y = sd(Y), CV_Y = sd(Y)/mean(Y),
                      cor_Pearson = cor(X, Y)))
}




# Profiled negative log-likelihood under H0 (shared m with delta shift)
nll0_profile <- function(par, x, y, delta) {
  lsX <- par[1]; lsY <- par[2]; zr <- par[3]
  sX  	<- exp(lsX)
  sY  	<- exp(lsY)
  rho 	<- tanh(zr)

  # profiled m-hat given (sX,sY) and delta
  mhat <- 0.5 * (mean(x)/sX + mean(y)/sY - delta)

  # standardized residuals
  rx <- x/sX - mhat
  ry <- y/sY - (mhat + delta)

  one_m_r2 <- 1 - rho^2
  n <- length(x)

  quad <- sum(rx^2 - 2*rho*rx*ry + ry^2) / one_m_r2
  logdet <- 2*log(sX) + 2*log(sY) + log(one_m_r2)

  0.5 * (n*log(2*pi) + n*logdet + quad)
}

# Profiled negative log-likelihood under H1 (separate means)
nll1_profile <- function(par, x, y) {
  lsX <- par[1]; lsY <- par[2]; zr <- par[3]
  sX  	<- exp(lsX)
  sY  	<- exp(lsY)
  rho 	<- tanh(zr)

  # profiled mX-hat, mY-hat
  mXhat <- mean(x)/sX
  mYhat <- mean(y)/sY

  rx <- x/sX - mXhat
  ry <- y/sY - mYhat

  one_m_r2 <- 1 - rho^2
  n <- length(x)

  quad <- sum(rx^2 - 2*rho*rx*ry + ry^2) / one_m_r2
  logdet <- 2*log(sX) + 2*log(sY) + log(one_m_r2)

  0.5 * (n*log(2*pi) + n*logdet + quad)
}



#Part 2: Simmulation ###################################################################################

# 2. Set up the parallel backend
# Detect the number of available cores and use them (leaving one free for system tasks)
n_cores <- detectCores()
if (n_cores > 1) {
  # Create a cluster
  cl <- makeCluster(n_cores - 1)
  registerDoParallel(cl)
  cat(paste("Registered parallel backend with", n_cores - 1, "cores.\n"))
} else {
  registerDoSEQ() # Fallback to sequential if only one core is available
  cat("Only one core detected, running sequentially.\n")
}

# 3. Run the long simulation in parallel

nsim <- 1000 # A large number for a long simulation
true_delta <- 0   # Model 1: eta_Y - eta_X = 0
cat(paste("Starting", nsim, "simulations...\n"))

n_sample_size=c(5, 10) #Sample Size
#n_sample_size=c(30, 100, 500, 1000) #Sample Size
rho_correlation=c(-.7,-.3, 0, .3, .7) #Correlation
#nsim=2 #Number of Simulation

Out_Matrix<-matrix(NA, nrow=20, ncol=6)


t0<-proc.time() # Start timing

for(i1 in 1:length(rho_correlation)){
    for(i2 in 1:length(n_sample_size)){


        # The %dopar% operator tells foreach to run the iterations in parallel
       results <- foreach(kk = 1:nsim, .combine = rbind) %dopar% {

    n <- n_sample_size[i2]

    set <- gen_two_normals_sameCV(n = n_sample_size[i2], mu1 = 5, mu2 = 8, CV = 0.4,
                                  rho = rho_correlation[i1], seed = (2025 + kk))
    xt <- set$X
    yt <- set$Y

    # good starting values from sample moments
    init_sX  <- sd(xt)
    init_sY  <- sd(yt)
    init_rho <- cor(xt, yt)
    par0 <- c(log(init_sX), log(init_sY), atanh(pmax(pmin(init_rho, 0.99), -0.99)))

    crit <- qchisq(.95, 1)
    true_delta <- 0   # Model 1: eta_Y - eta_X = 0

    # H1:
    op1 <- optim(par0, nll1_profile, x = xt, y = yt, method = "Nelder-Mead",
                 control = list(maxit = 4000, reltol = 1e-10))

    # LRT(delta) - crit; if it fails: NA
    g <- function(d) {
      op0 <- optim(par0, nll0_profile, x = xt, y = yt, delta = d, method = "Nelder-Mead",
                   control = list(maxit = 4000, reltol = 1e-10))
      if (op0$convergence != 0) return(NA)
      2 * (op0$value - op1$value) - crit
    }

    # MLE of delta (= eta_Y - eta_X)
    sXm <- sqrt(mean((xt - mean(xt))^2)); sYm <- sqrt(mean((yt - mean(yt))^2))
    d_hat <- mean(yt)/sYm - mean(xt)/sXm

    find_end <- function(dir, max_step = 2^10) {
      step <- 1
      repeat {
        b  <- d_hat + dir * step
        gb <- g(b)
        if (is.na(gb)) return(NA)
        if (gb > 0) break
        step <- 2 * step
        if (step > max_step) return(NA)
      }
      uniroot(g, sort(c(d_hat, b)), tol = 1e-6)$root
    }

    Lower_CI <- find_end(-1)
    Upper_CI <- find_end(+1)

    if (op1$convergence != 0 || is.na(Lower_CI) || is.na(Upper_CI)) {
      data.frame(sim_id = kk, Cov.out = NA, Len_out = NA,
                 Lower = NA, Upper = NA, fail = 1)
    } else {
      data.frame(sim_id = kk,
                 Cov.out = as.numeric(Lower_CI <= true_delta & true_delta <= Upper_CI),
                 Len_out = Upper_CI - Lower_CI,
                 Lower = Lower_CI, Upper = Upper_CI, fail = 0)
    }
} # End of foreach

        Cov_rate_out<-results[,2]
        Length_CI_out<-results[,3]
        Cov_rate_out[is.infinite(Cov_rate_out)]<-NA
        Length_CI_out[is.infinite(Length_CI_out)]<-NA
        Cov_rate_out<-na.omit(Cov_rate_out)
        Length_CI_out<-na.omit(Length_CI_out)


        ok <- results$fail == 0                      # only successes
        cov_i  <- results$Cov.out[ok]
        len_i  <- results$Len_out[ok]
        low_i  <- results$Lower[ok]
        up_i   <- results$Upper[ok]
        R      <- sum(ok)                            # number of "s"

        OM <- (i1 - 1) * length(n_sample_size) + i2
       Out_Matrix[OM, 1] <- mean(cov_i)                              # coverage
       Out_Matrix[OM, 2] <- round(mean(len_i), 4)                    # ave length
       Out_Matrix[OM, 3] <- sum(results$fail)                        # number of "f"
       Out_Matrix[OM, 4] <- round(sqrt(mean(cov_i) * (1 - mean(cov_i)) / R), 4)  # coverage의 MC SE
       Out_Matrix[OM, 5] <- mean(true_delta < low_i)                 # lower miss
       Out_Matrix[OM, 6] <- mean(true_delta > up_i)                  # upper miss

    }
}



# 4. Stop the cluster and analyze results
if (exists("cl")) {
  stopCluster(cl) # Important: always stop the cluster when done
}


t1<-proc.time() # Stop timing

t1-t0
Out_Matrix

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Registered parallel backend with 7 cores.
Starting 1000 simulations...


   user  system elapsed 
  6.804   0.848 134.610 

0.876,6.0672,0,0.0104,0.066,0.058
0.928,3.6361,0,0.0082,0.039,0.033
0.858,6.6308,0,0.0110,0.062,0.080
0.911,3.9241,0,0.0090,0.049,0.040
0.856,6.3801,0,0.0111,0.067,0.077
0.911,3.8952,0,0.0090,0.055,0.034
0.835,6.1305,0,0.0117,0.079,0.086
0.908,3.7205,0,0.0091,0.055,0.037
0.833,5.1545,0,0.0118,0.085,0.082
0.917,3.0220,0,0.0087,0.049,0.034
NA,NA,NA,NA,NA,NA
